# Cutouts with Hyrax

This notebook presents the two steps to produce image cutouts that can be used with Hyrax.

Step one is producing a catalog of ra/decs. Here we use the LSST Butler, but note that using LSDB is an excellent option as well.

Step two is to provide the ra/dec catalog to Hyrax to process in one of two ways, either:
- creating and caching the image cutouts to disk
- retrieving the cutouts from disk in real time

In [1]:
catalog_file_directory = "DIRECTORY-WHERE-I-SAVE-MY-CATALOG" # "/astro/store/shiren/lincc_frameworks"
saved_cutouts_directory = "DIRECTORY-WHERE-I-SAVE-MY-CUTOUTS" # "/astro/store/shiren/lincc_frameworks/hyrax_cutout_scratch" 

In [2]:
import lsdb
import pickle
import hyrax
from lsst.daf.butler import Butler
import numpy as np
import pandas as pd
import astropy.units as u
from IPython.display import Image
import pylab as plt
from astropy.table import Table, hstack, vstack

## Produce the ra/dec catalog

In the next cells, we'll instantiate a Butler instance to query. The result will be about 2.1M objectIds with corresponding ra and dec values.

In [3]:
collection = "LSSTCam/runs/DRP/DP2"
butler_repo = "/astro/store/shire/hyrax-coadd-dp2/"
butler_skymap = "lsst_cells_v2"

# Create an instance of the butler and get all the available collections
butler = Butler(butler_repo, collections=collection)
all_collections = list(butler.registry.queryCollections())

In [4]:
# Expand this list to include any of the available columns of interest that 
# might help with filtering for your science use case
INCOLS = [
    'objectId',
    'coord_ra',
    'coord_dec',
]

In [5]:
# Here we query the butler for data

object_tables = []

for tract in [5063,4849,4848]:
    object_table = butler.get(
        'object',
        dataId={'skymap': butler_skymap, 'tract': tract},
        collections=all_collections,
        parameters={"columns":INCOLS}
    )
    object_table.meta = {}  # Hack to get rid of conflicts
    
    # >>> Additional filtering and processing based on the tabular data can happen here <<<    
    object_tables.append(object_table)

ecdfs_objects = vstack(object_tables,join_type='exact',metadata_conflicts='warn')

In [6]:
ecdfs_objects

objectId,coord_ra,coord_dec
int64,float64,float64
755368611450718588,53.89225401671883,-28.264414861298743
755368611450718599,53.88232012169452,-28.264216416143928
755368611450718605,53.899424108157625,-28.26360885496538
755368611450718612,53.90958843474439,-28.263466230504132
755368611450718628,53.86043353277437,-28.263185044822965
755368611450718632,53.89839031795252,-28.26260328732315
755368611450718636,53.906880610667926,-28.262591618352843
755368611450718641,53.905924949631384,-28.262483801743954
755368611450718652,53.85497523432888,-28.262398242385256


## Save the catalog
This is just a demo, so we don't want to save 2.1M rows - the resulting file would be several gigabytes in size. Instead, we'll restrict ourselves to only the first N rows of the table.

In [7]:
# Only export the N rows from the dataframe for demonstration purposes.
# Under normal circumstances, we would save the entire dataframe.
num_samples = 10
catalog_file_path = f"{catalog_file_directory}/my_catalog_{num_samples}.pkl"

with open(catalog_file_path,"wb") as f:
    pickle.dump(ecdfs_objects[0:num_samples], f)

## Producing image cutouts

We can use the catalog file that was just created to tell Hyrax which cutouts to create.
We'll create an instance of Hyrax, update it's configuration settings as needed, then produce the cutouts.

In [8]:
h = hyrax.Hyrax()

# These configs will let Hyrax know how to access the Butler to create the cutouts
h.config["general"]["data_dir"] = saved_cutouts_directory
h.config["data_set"]["astropy_table"] = catalog_file_path
h.config["data_set"]["butler_repo"] = butler_repo
h.config["data_set"]["butler_collection"] = all_collections
h.config["data_set"]["skymap"] = butler_skymap

# These configs define specifics for the cutouts 
h.config["data_set"]["semi_width_deg"] = (17 * u.arcsec).to(u.deg).value
h.config["data_set"]["semi_height_deg"] = (17 * u.arcsec).to(u.deg).value
h.config["data_set"]["filters"] = ["u","g","r","i","z","y"]
h.config["data_set"]["object_id_column_name"] = "objectId"
h.config["data_set"]["use_cache"] = False
h.config["data_set"]["crop_to"] = [150,150]   #NO IMPACT ON DOWNLOAD. TO PREVENT CONFIG COMPLAINS

### Option 1 - Cache cutouts to disk

With this option, the cutouts will be created in memory, then saved to disk for later use.
Note that this process can be time consuming as the number of cutouts increases. We attempt to optimize this by minimizing the number of times a large image file needs to be read from disk, but disk I/O is always going to be a limiting factor here. 

In [9]:
# We create a data_request to tell Hyrax what tool to use to create the image cutouts
data_request = {
    "train":{
        "data": {
            "dataset_class": "DownloadedLSSTDataset",
            "data_location": saved_cutouts_directory,
            "primary_id_field": "objectId",
            "fields": ["image"],
        }
    },
}
h.set_config('data_request', data_request)

[2026-08-06 16:24:18,693 hyrax.config_utils:WARNING] Runtime config contains key or section 'butler_repo' which has no default defined. All configuration keys and sections must be defined in /astro/store/shiren/mtauraso/miniforge3/envs/rubin-edp2-workshop/lib/python3.13/site-packages/hyrax/hyrax_default_config.toml
[2026-08-06 16:24:18,693 hyrax.config_utils:WARNING] Runtime config contains key or section 'butler_collection' which has no default defined. All configuration keys and sections must be defined in /astro/store/shiren/mtauraso/miniforge3/envs/rubin-edp2-workshop/lib/python3.13/site-packages/hyrax/hyrax_default_config.toml
[2026-08-06 16:24:18,693 hyrax.config_utils:WARNING] Runtime config contains key or section 'skymap' which has no default defined. All configuration keys and sections must be defined in /astro/store/shiren/mtauraso/miniforge3/envs/rubin-edp2-workshop/lib/python3.13/site-packages/hyrax/hyrax_default_config.toml


In [10]:
a = h.prepare()

[2026-08-06 16:24:18,723 hyrax.datasets.downloaded_lsst_dataset:INFO] Creating new manifest
[2026-08-06 16:24:18,728 hyrax.datasets.downloaded_lsst_dataset:INFO] Initialized new manifest at /astro/store/shiren/lincc_frameworks/hyrax_cutout_scratch/manifest.fits
[2026-08-06 16:24:18,728 hyrax.verbs.prepare:INFO] Finished Prepare


In [11]:
dl_lsst_dataset_instance = a['train'].prepped_datasets['data']

### Create the cutouts
This is the step that actually creates the image cutouts. Feel free to open a terminal and watch as the cutouts land in the directory.

In [ ]:
manifest = dl_lsst_dataset_instance.download_cutouts(max_workers=10)

[2026-08-06 16:24:18,737 hyrax.datasets.downloaded_lsst_dataset:INFO] Syncing manifest with filesystem...
[2026-08-06 16:24:18,738 hyrax.datasets.downloaded_lsst_dataset:INFO] Downloading 10 cutouts using 10 threads.

### Option 2 - Create cutouts on the fly
In this example, we use a different dataset_class, `RangeReadLSSTDataset` to read only the required pixels into memory. 
Note that we will not persist the cutouts to disk with this dataset_class.
Because we only read the requested pixels into memory, this dataset class tends to run much faster than the previous example.

You should try experimenting with both to see which is more effective for your particular workflow.
If you would like some guidance, ask a LINCC Frameworks engineer!

In [ ]:
# As before, we create a data_request to tell Hyrax what tool to use to create the image cutouts
data_request = {
    "train":{
        "data": {
            "dataset_class": "RangeReadLSSTDataset",  # <-- Note, we're using `RangeReadLSSTDataset` here
            "data_location": saved_cutouts_directory,
            "primary_id_field": "objectId",
            "fields": ["image"],
        }
    },
}
h.set_config('data_request', data_request)

### Train a simple model
Before we read in large files, created cutouts and then save them to disk. 
Here we'll simply train a model directly (but poorly, recall we're only using 10 cutouts)

In [ ]:
h.set_config('model.name', 'HyraxAutoencoderV2')
h.set_config('data_loader.batch_size', 64)
h.train()